# 🎁 Bônus da Semana 11 — CRUD no Python: Integração Python e PostgreSQL

Este notebook é um **bônus opcional**, separado do notebook principal desta semana (`notebook_colab_aluno.ipynb`, que usa `pyodbc` + SQLite).

Aqui você repete exatamente a mesma lógica de conexão — **conectar → criar cursor → executar comandos SQL → encerrar** — mas contra um banco de verdade que você já criou: o **PostgreSQL**, no banco `northwind` (o mesmo que você usa desde a Semana 08 — 91 clientes, 830 pedidos, 77 produtos reais, com clientes do Brasil incluídos).

**⚠️ Pré-requisito:** o PostgreSQL precisa estar rodando na sua máquina, e o banco `northwind` já precisa existir (criado nas Semanas 08/09). Troque `SUA_SENHA_AQUI` pela senha que você mesmo configurou no PostgreSQL.

## Connect — a mesma lógica, biblioteca diferente

Onde no notebook principal você usava `pyodbc.connect(...)` com uma string cheia de `Driver={...}`, o `psycopg2` (a biblioteca oficial do PostgreSQL) usa parâmetros nomeados diretos — mais simples, porque foi feita só para esse banco (não precisa de driver ODBC nenhum).

In [ ]:
import psycopg2

conexao = psycopg2.connect(
    host="localhost",
    dbname="northwind",
    user="postgres",
    password="SUA_SENHA_AQUI",
)
cursor = conexao.cursor()
print("Conexão bem sucedida")

## Read — consultando com parâmetro seguro

A diretoria da Northwind quer saber quais clientes são do Brasil. Repare no `%s` dentro do SQL, no lugar do valor `'Brazil'` — é assim que o `psycopg2` recebe parâmetros: o valor vai numa tupla separada, nunca colado direto na string.

**Isso não é o mesmo `%s` de formatação de texto do Python** (tipo `"Olá, %s" % nome`), mesmo parecendo igual. Aqui, o `%s` é um espaço reservado que o `psycopg2` interpreta sozinho, mandando o valor pro banco separado do comando SQL — é exatamente essa separação que impede o SQL Injection (ver abaixo). Se fosse formatação comum do Python, o valor viraria texto colado no comando, e o problema de segurança voltaria a existir.

**Por que a tupla `("Brazil",)` tem uma vírgula depois de um valor só?** Em Python, `("Brazil")` sem vírgula NÃO é uma tupla — é só a string `"Brazil"` entre parênteses (os parênteses aqui não fariam nada). A vírgula depois do valor é o que avisa o Python "isto é uma tupla de 1 item", mesmo parecendo sobrando à primeira vista.

**Por que não colar o valor direto no texto?** Colar valores direto na string de SQL (`f"... WHERE country = '{pais}'"`) abre brecha pra *SQL Injection* — alguém pode digitar um valor malicioso que muda o comando inteiro. Passar o valor como parâmetro (`%s` + tupla) faz o `psycopg2` tratar esse valor sempre como um dado puro, nunca como parte do comando.

In [ ]:
cursor.execute("SELECT company_name, city, country FROM customers WHERE country = %s", ("Brazil",))
clientes_brasil = cursor.fetchall()
print(clientes_brasil)

A mesma consulta, agora com `pd.read_sql()` — o resultado já pronto como DataFrame. Uma observação: o Pandas mostra um aviso amarelo (`UserWarning`) dizendo que só testa oficialmente conexões `sqlite3` ou SQLAlchemy — pode ignorar, o resultado sai correto do mesmo jeito.

In [ ]:
import pandas as pd

tabela_brasil = pd.read_sql(
    "SELECT company_name, city, country FROM customers WHERE country = %s",
    conexao,
    params=("Brazil",),
)
display(tabela_brasil)

## Create — cadastrando um cliente novo

**Contextualização:** a Squad Estúdio virou cliente novo da Northwind e precisa ser cadastrada. Diferente do `chinook.db` (onde o Id era automático), a tabela `customers` do Northwind usa um código de 5 letras escolhido na hora do cadastro (como `'ALFKI'`, `'ANATR'`).

In [ ]:
cursor.execute(
    "INSERT INTO customers (customer_id, company_name, contact_name, city, country) VALUES (%s, %s, %s, %s, %s)",
    ("SQUAD", "Squad Estúdio", "Ana Souza", "Curitiba", "Brazil"),
)
conexao.commit()

cursor.execute("SELECT customer_id, company_name, city FROM customers WHERE customer_id = %s", ("SQUAD",))
print(cursor.fetchall())

## Update — a Squad Estúdio mudou de cidade

**Contextualização:** a Squad Estúdio avisou que se mudou para São Paulo.

In [ ]:
cursor.execute(
    "UPDATE customers SET city = %s WHERE customer_id = %s",
    ("São Paulo", "SQUAD"),
)
conexao.commit()

cursor.execute("SELECT customer_id, city FROM customers WHERE customer_id = %s", ("SQUAD",))
print(cursor.fetchall())

## Um JOIN de verdade — o que o SQLite sozinho não mostrava tão bem

O Northwind tem tabelas de verdade relacionadas entre si (`customers` → `orders` → `order_details` → `products`). Isso permite uma pergunta que nenhuma tabela isolada responde: **quais produtos um cliente específico já comprou?**

Repare que, no código abaixo, cada tabela ganha um **alias** — um apelido curto (`c` para `customers`, `o` para `orders`, `od` para `order_details`, `p` para `products`) — escrito logo depois do nome da tabela (`customers c`, `orders o`). Isso existe só pra não repetir o nome inteiro da tabela toda vez que uma coluna precisa dizer de onde ela vem (`c.company_name`, `o.customer_id`) — numa consulta com 4 tabelas juntas, sem alias o SQL ficaria bem mais longo e repetitivo.

In [ ]:
cursor.execute('''
    SELECT c.company_name, p.product_name, od.quantity
    FROM customers c
    JOIN orders o ON o.customer_id = c.customer_id
    JOIN order_details od ON od.order_id = o.order_id
    JOIN products p ON p.product_id = od.product_id
    WHERE c.company_name = %s
    ORDER BY p.product_name
''', ("Hanari Carnes",))

print(cursor.fetchall())

## Fato x Dimensão — por que o Northwind é um bom exemplo pra isso

Repare que a última consulta juntou 4 tabelas, e cada uma tem um papel diferente:

| Tabela | Papel | O que ela guarda |
|---|---|---|
| `order_details` | **Fato** | Cada linha é uma transação de verdade: um produto, dentro de um pedido, com quantidade/preço/desconto |
| `customers` | Dimensão | Descreve QUEM comprou |
| `products` | Dimensão | Descreve O QUE foi comprado |
| `orders` | Dimensão (e liga ao cliente) | Descreve QUANDO e para onde foi o pedido |

**Tabela fato** é aquela cujas linhas representam eventos/transações que aconteceram (aqui, cada item vendido) — ela concentra os números que você soma/conta (`quantity`, `unit_price`, `discount`). **Tabelas de dimensão** são as que descrevem, com texto, os "quem/o quê/onde" em volta de cada fato — elas não mudam a cada venda, só são consultadas pra dar contexto ao fato.

Essa distinção é o que torna possível somar receita "por categoria" ou "por país" sem precisar de uma tabela nova pra cada pergunta — você só troca a dimensão que entra no `JOIN`.

### Receita por categoria (Fato + 2 Dimensões)

**Contextualização:** a diretoria quer saber qual categoria de produto gera mais receita, considerando os descontos já aplicados.

A fórmula de receita de cada item é `unit_price × quantity × (1 - discount)` — está em `order_details` (o fato); a categoria de cada produto vem de duas dimensões encadeadas (`products` → `categories`).

Duas coisas novas no código abaixo:
- **`::numeric`** — o PostgreSQL calcula a soma como um número de ponto flutuante, que pode vir com muitas casas decimais estranhas; `::numeric` converte esse resultado pra um tipo numérico exato antes do `ROUND()` arredondar certinho em 2 casas.
- **`ORDER BY receita`** — `receita` não é uma coluna de nenhuma tabela, é o nome que a própria consulta deu pro resultado da soma (logo depois de `AS`). O PostgreSQL permite usar esse nome no `ORDER BY` da mesma consulta que o criou.

In [ ]:
cursor.execute('''
    SELECT cat.category_name,
           ROUND(SUM(od.unit_price * od.quantity * (1 - od.discount))::numeric, 2) AS receita
    FROM order_details od
    JOIN products p ON p.product_id = od.product_id
    JOIN categories cat ON cat.category_id = p.category_id
    GROUP BY cat.category_name
    ORDER BY receita DESC
''')

for linha in cursor.fetchall():
    print(linha)

## WHERE x HAVING — filtrando antes ou depois de agrupar

Os dois filtram linhas, mas em momentos diferentes da consulta:

| | Quando filtra | Pode usar `SUM()`, `COUNT()`, etc.? |
|---|---|---|
| `WHERE` | Antes do `GROUP BY` — filtra linhas individuais, cruas | Não |
| `HAVING` | Depois do `GROUP BY` — filtra grupos já agregados | Sim |

**`WHERE`** — quantos pedidos foram feitos depois de 01/01/1998 (filtra linhas de `orders`, sem agrupar nada):

In [ ]:
cursor.execute("SELECT COUNT(*) FROM orders WHERE order_date > %s", ("1998-01-01",))
print("Pedidos depois de 1998-01-01:", cursor.fetchall())

**`HAVING`** — quais categorias têm receita total acima de R$ 150.000. Isso é impossível de escrever com `WHERE`, porque `SUM(...)` só existe DEPOIS que as linhas já foram agrupadas por categoria — é a mesma consulta de receita por categoria de cima, só que agora filtrando o resultado já somado:

In [ ]:
cursor.execute('''
    SELECT cat.category_name,
           ROUND(SUM(od.unit_price * od.quantity * (1 - od.discount))::numeric, 2) AS receita
    FROM order_details od
    JOIN products p ON p.product_id = od.product_id
    JOIN categories cat ON cat.category_id = p.category_id
    GROUP BY cat.category_name
    HAVING SUM(od.unit_price * od.quantity * (1 - od.discount)) > 150000
    ORDER BY receita DESC
''')

for linha in cursor.fetchall():
    print(linha)

### Mais um relacionamento: receita por país

**Contextualização:** agora a diretoria quer saber quais países mais compram, pra decidir onde reforçar o time comercial. Mesma tabela fato (`order_details`), dimensão diferente (`customers`, através de `orders`).

In [ ]:
cursor.execute('''
    SELECT c.country,
           ROUND(SUM(od.unit_price * od.quantity * (1 - od.discount))::numeric, 2) AS receita
    FROM order_details od
    JOIN orders o ON o.order_id = od.order_id
    JOIN customers c ON c.customer_id = o.customer_id
    GROUP BY c.country
    ORDER BY receita DESC
    LIMIT 5
''')

for linha in cursor.fetchall():
    print(linha)

## Limpeza de dados — nem todo dado real está pronto pra usar

Diferente de um exercício didático, o `northwind` tem casos reais que exigem decisão antes de analisar. Dois exemplos concretos deste banco:

1. **Pedidos sem `shipped_date`** — significa que o pedido ainda não foi despachado, não que o dado está "quebrado". Se você calculasse um tempo médio de entrega sem excluir essas linhas, o resultado sairia errado.
2. **Produtos descontinuados** (`discontinued = 1`) — ainda aparecem em pedidos antigos, mas não deveriam entrar numa análise de "catálogo ativo hoje".

In [ ]:
cursor.execute("SELECT COUNT(*) FROM orders WHERE shipped_date IS NULL")
print("Pedidos ainda não despachados:", cursor.fetchall())

cursor.execute("SELECT order_id, order_date FROM orders WHERE shipped_date IS NULL ORDER BY order_id LIMIT 5")
print("Exemplos:", cursor.fetchall())

Antes de calcular uma média de tempo de entrega, o filtro `WHERE shipped_date IS NOT NULL` remove exatamente esses pedidos pendentes — sem isso, a métrica mistura "ainda não chegou" com "demorou muito".

**Sobre `shipped_date - order_date` no código abaixo:** no PostgreSQL, subtrair uma data de outra devolve direto o número de dias entre elas (aqui, um `AVG()` dessa diferença dá a média de dias até o envio) — não precisa converter nada antes, diferente de subtrair dois textos ou dois números de tipos diferentes.

In [ ]:
cursor.execute('''
    SELECT ROUND(AVG(shipped_date - order_date), 1) AS media_dias_entrega
    FROM orders
    WHERE shipped_date IS NOT NULL
''')
print("Média de dias até o envio (só pedidos já despachados):", cursor.fetchall())

E antes de contar quantos produtos existem "hoje" no catálogo, o filtro `discontinued = 0` remove os produtos que a Northwind já parou de vender:

In [ ]:
cursor.execute("SELECT COUNT(*) FROM products")
print("Total de produtos (com descontinuados):", cursor.fetchall())

cursor.execute("SELECT COUNT(*) FROM products WHERE discontinued = 0")
print("Produtos ativos no catálogo:", cursor.fetchall())

## Delete — encerrando o cadastro de teste

**Contextualização:** o cadastro da Squad Estúdio foi só um teste — hora de remover.

In [ ]:
cursor.execute("DELETE FROM customers WHERE customer_id = %s", ("SQUAD",))
conexao.commit()

cursor.execute("SELECT * FROM customers WHERE customer_id = %s", ("SQUAD",))
print("Deve ficar vazio:", cursor.fetchall())

cursor.close()
conexao.close()

## ✏️ Atividade Bônus — Sua vez

**Contextualização:** o time de compras quer saber quais categorias de produto existem no catálogo, para decidir onde focar a próxima campanha.

**Comando:** conecte no banco `northwind` e consulte todas as linhas de `category_name` da tabela `categories`. Depois, feche a conexão.

In [ ]:
# escreva seu código aqui

## 🇧🇷 Traduzindo as tabelas e colunas para português

**⚠️ Esta seção altera o banco `northwind` de verdade, de forma permanente** — diferente de tudo que veio antes (que sempre limpou o que criou), o `ALTER TABLE ... RENAME` não tem "desfazer" fácil. Só rode isto se você quiser mesmo deixar seu `northwind` local em português dali pra frente. Se algo der errado ou você quiser voltar ao inglês, é preciso recriar o banco do zero com o `northwind.sql` original.

Até aqui, todo nome de tabela e coluna esteve em inglês (o Northwind original é americano). O comando `ALTER TABLE ... RENAME TO` renomeia uma tabela; `ALTER TABLE ... RENAME COLUMN ... TO ...` renomeia uma coluna dela — os dados dentro não mudam em nada, só o nome de quem os guarda.

**Por que uma conexão nova (`conexaoRenomear`) em vez da `conexao` de sempre?** Porque a célula do Delete, lá em cima, já fechou a `conexao` e o `cursor` originais (`conexao.close()`) — uma vez fechada, uma conexão não pode ser reaberta, então esta seção abre a sua própria, com nomes diferentes só pra deixar claro que é uma conexão separada.

In [ ]:
import psycopg2

conexaoRenomear = psycopg2.connect(
    host="localhost",
    dbname="northwind",
    user="postgres",
    password="SUA_SENHA_AQUI",
)
cursor2 = conexaoRenomear.cursor()

cursor2.execute("ALTER TABLE customers RENAME TO clientes")
cursor2.execute("ALTER TABLE clientes RENAME COLUMN customer_id TO id_cliente")
cursor2.execute("ALTER TABLE clientes RENAME COLUMN company_name TO nome_empresa")
cursor2.execute("ALTER TABLE clientes RENAME COLUMN contact_name TO nome_contato")
cursor2.execute("ALTER TABLE clientes RENAME COLUMN city TO cidade")
cursor2.execute("ALTER TABLE clientes RENAME COLUMN country TO pais")

cursor2.execute("ALTER TABLE orders RENAME TO pedidos")
cursor2.execute("ALTER TABLE pedidos RENAME COLUMN order_id TO id_pedido")
cursor2.execute("ALTER TABLE pedidos RENAME COLUMN customer_id TO id_cliente")
cursor2.execute("ALTER TABLE pedidos RENAME COLUMN order_date TO data_pedido")
cursor2.execute("ALTER TABLE pedidos RENAME COLUMN shipped_date TO data_envio")

cursor2.execute("ALTER TABLE order_details RENAME TO itens_pedido")
cursor2.execute("ALTER TABLE itens_pedido RENAME COLUMN order_id TO id_pedido")
cursor2.execute("ALTER TABLE itens_pedido RENAME COLUMN product_id TO id_produto")
cursor2.execute("ALTER TABLE itens_pedido RENAME COLUMN unit_price TO preco_unitario")
cursor2.execute("ALTER TABLE itens_pedido RENAME COLUMN quantity TO quantidade")
cursor2.execute("ALTER TABLE itens_pedido RENAME COLUMN discount TO desconto")

cursor2.execute("ALTER TABLE products RENAME TO produtos")
cursor2.execute("ALTER TABLE produtos RENAME COLUMN product_id TO id_produto")
cursor2.execute("ALTER TABLE produtos RENAME COLUMN product_name TO nome_produto")
cursor2.execute("ALTER TABLE produtos RENAME COLUMN category_id TO id_categoria")
cursor2.execute("ALTER TABLE produtos RENAME COLUMN discontinued TO descontinuado")

cursor2.execute("ALTER TABLE categories RENAME TO categorias")
cursor2.execute("ALTER TABLE categorias RENAME COLUMN category_id TO id_categoria")
cursor2.execute("ALTER TABLE categorias RENAME COLUMN category_name TO nome_categoria")

conexaoRenomear.commit()
print("Tabelas e colunas renomeadas para português.")

Confirme rodando a mesma pergunta do JOIN lá de cima ("quais produtos a Hanari Carnes comprou"), agora só com nomes em português:

In [ ]:
cursor2.execute('''
    SELECT cli.nome_empresa, prod.nome_produto, ip.quantidade
    FROM clientes cli
    JOIN pedidos ped ON ped.id_cliente = cli.id_cliente
    JOIN itens_pedido ip ON ip.id_pedido = ped.id_pedido
    JOIN produtos prod ON prod.id_produto = ip.id_produto
    WHERE cli.nome_empresa = %s
    ORDER BY prod.nome_produto
    LIMIT 5
''', ("Hanari Carnes",))

print(cursor2.fetchall())

cursor2.close()
conexaoRenomear.close()

### ✅ O que você fez neste bônus

- Conectou o Python ao PostgreSQL com `psycopg2` (a mesma lógica do `pyodbc`, biblioteca diferente).
- Usou parâmetros `%s` pra evitar SQL Injection, e entendeu por que isso não é a mesma coisa da formatação de texto do Python.
- Fez o CRUD completo (Create, Read ×2, Update, Delete) num banco relacional de verdade.
- Rodou JOINs reais de várias tabelas usando alias, incluindo agregações (receita por categoria, receita por país).
- Entendeu a diferença entre **tabela fato** (`order_details`, os eventos/transações) e **tabelas de dimensão** (`customers`, `products`, `categories`, o contexto descritivo em volta de cada fato).
- Diferenciou `WHERE` (filtra linhas antes de agrupar) de `HAVING` (filtra grupos já agregados).
- Praticou limpeza de dados real: excluir pedidos ainda não despachados antes de calcular tempo de entrega, e excluir produtos descontinuados antes de contar o catálogo ativo.
- Traduziu tabelas e colunas do Northwind pra português com `ALTER TABLE ... RENAME`.